# Analyse Exploratoire des Données (EDA)
## Projet : Prédiction d'inclusion bancaire — Afrique de l'Est

---

### Objectif
Ce notebook réalise l'analyse exploratoire du dataset **Financial Inclusion** issu des enquêtes FinScope.

Il couvre les étapes suivantes :
1. Chargement et aperçu du dataset brut
2. Analyse de la structure et des types de données
3. Détection et traitement des valeurs manquantes
4. Détection des doublons
5. Filtrage des années hors période d'enquête
6. Normalisation des noms de colonnes
7. Cible inconnue et niveau d'éducation sans libellé
8. Cohérence entre pays : constats, et libellés ville / campagne corrigés en Tanzanie
9. Contrôles qualité automatiques et export du dataset nettoyé

---

**Source** : `data/Financial_inclusion_dataset.csv`  
**Output** : `data/financial_inclusion_clean.csv`

---
## 0. Installation des dépendances

In [1]:
%pip install matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: C:\Users\marcs\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


---
## 1. Chargement du dataset

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH  = "../data/Financial_inclusion_dataset.csv"
OUTPUT_PATH = "../data/financial_inclusion_clean.csv"
SURVEY_YEARS = [2016, 2017, 2018]

df = pd.read_csv(DATA_PATH)

print(f"Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

Dataset chargé : 23524 lignes × 13 colonnes


,country,year,uniqueid,Has a Bank account,Type of Location,Cell Phone Access,household_size,Respondent Age,gender_of_respondent,The relathip with head,marital_status,Level of Educuation,Type of Job
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3.0,24.0,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5.0,70.0,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5.0,26.0,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5.0,34.0,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8.0,26.0,Male,Child,Single/Never Married,Primary education,Informally employed


---
## 2. Structure et types de données

In [3]:
# Dimensions du dataset
print(f"Dimensions : {df.shape}\n")

# Types de colonnes et valeurs non-nulles
print("=== Informations générales ===")
df.info()

print("\n=== Statistiques descriptives (colonnes numériques) ===")
df.describe()

Dimensions : (23524, 13)

=== Informations générales ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country                 23510 non-null  object 
 1   year                    23524 non-null  int64  
 2   uniqueid                23524 non-null  object 
 3   Has a Bank account      23488 non-null  object 
 4   Type of Location        23509 non-null  object 
 5   Cell Phone Access       23513 non-null  object 
 6   household_size          23496 non-null  float64
 7   Respondent Age          23490 non-null  float64
 8   gender_of_respondent    23490 non-null  object 
 9   The relathip with head  23520 non-null  object 
 10  marital_status          23492 non-null  object 
 11  Level of Educuation     23495 non-null  object 
 12  Type of Job             23494 non-null  object 
dtypes: float64(2), int64(1), object(10

,year,household_size,Respondent Age
count,23524.000000,23496.000000,23490.000000
mean,2016.979000,3.681818,38.804300
std,0.899669,2.279933,16.519996
min,2016.000000,0.000000,16.000000
25%,2016.000000,2.000000,26.000000
50%,2017.000000,3.000000,35.000000
75%,2018.000000,5.000000,49.000000
max,2056.000000,21.000000,100.000000


In [4]:
# Séparation colonnes numériques / catégorielles
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Colonnes numériques  ({len(num_cols)}) : {num_cols}")
print(f"Colonnes catégorielles ({len(cat_cols)}) : {cat_cols}")

Colonnes numériques  (3) : ['year', 'household_size', 'Respondent Age']
Colonnes catégorielles (10) : ['country', 'uniqueid', 'Has a Bank account', 'Type of Location', 'Cell Phone Access', 'gender_of_respondent', 'The relathip with head', 'marital_status', 'Level of Educuation', 'Type of Job']


---
## 3. Analyse et traitement des valeurs manquantes

In [5]:
# Détection des valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "Valeurs manquantes": missing,
    "Pourcentage (%)": missing_pct
}).query("`Valeurs manquantes` > 0")

if missing_report.empty:
    print("✅ Aucune valeur manquante détectée.")
else:
    print("⚠️ Colonnes avec valeurs manquantes :")
    display(missing_report)

⚠️ Colonnes avec valeurs manquantes :

,Valeurs manquantes,Pourcentage (%)
country,14,0.06
Has a Bank account,36,0.15
Type of Location,15,0.06
Cell Phone Access,11,0.05
household_size,28,0.12
Respondent Age,34,0.14
gender_of_respondent,34,0.14
The relathip with head,4,0.02
marital_status,32,0.14
Level of Educuation,29,0.12


### Pays manquant : déduit de l'année

Chaque pays n'a été enquêté qu'une année : Rwanda 2016, Tanzanie 2017, Kenya et Ouganda 2018. Une ligne sans pays mais d'une année propre à un seul pays appartient à ce pays. L'identifiant le confirme : il est numéroté dans chaque pays, et celui de la ligne doit combler un trou de cette numérotation, sans doublon.

In [6]:
ids = df["uniqueid"].str.extract(r"(\d+)$")[0].astype(int)
countries_by_year = df.dropna(subset=["country"]).groupby("year")["country"].unique()
single_country_years = {
    year: countries[0]
    for year, countries in countries_by_year.items()
    if year in SURVEY_YEARS and len(countries) == 1
}

for year, country in single_country_years.items():
    rows = df["country"].isna() & (df["year"] == year)
    if not rows.any():
        continue
    country_ids = ids[df["country"] == country]
    assert ids[rows].between(country_ids.min(), country_ids.max()).all(), "identifiant hors de la plage du pays"
    assert not ids[rows].isin(country_ids).any(), "identifiant déjà attribué dans le pays"
    df.loc[rows, "country"] = country
    print(f"  {rows.sum()} ligne(s) de {year} sans pays → '{country}' (identifiants {ids[rows].min()} à {ids[rows].max()})")

print(f"\n✅ Pays encore manquants : {df['country'].isna().sum()}")

  14 ligne(s) de 2017 sans pays → 'Tanzania' (identifiants 3398 à 3514)

✅ Pays encore manquants : 0


In [7]:
# Traitement des valeurs manquantes
# - Colonnes numériques  → remplacement par la médiane (robuste aux outliers)
# - Colonnes catégorielles → remplacement par 'Unknown'

for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)
        print(f"  [NUM] '{col}' → médiane = {median_value}")

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna("Unknown")
        print(f"  [CAT] '{col}' → 'Unknown'")

# Vérification finale
remaining_nulls = df.isnull().sum().sum()
print(f"\n✅ Valeurs manquantes restantes : {remaining_nulls}")

  [NUM] 'household_size' → médiane = 3.0
  [NUM] 'Respondent Age' → médiane = 35.0
  [CAT] 'Has a Bank account' → 'Unknown'
  [CAT] 'Type of Location' → 'Unknown'
  [CAT] 'Cell Phone Access' → 'Unknown'
  [CAT] 'gender_of_respondent' → 'Unknown'
  [CAT] 'The relathip with head' → 'Unknown'
  [CAT] 'marital_status' → 'Unknown'
  [CAT] 'Level of Educuation' → 'Unknown'
  [CAT] 'Type of Job' → 'Unknown'



✅ Valeurs manquantes restantes : 0


---
## 4. Détection des doublons

In [8]:
duplicate_count = df.duplicated().sum()

if duplicate_count == 0:
    print("✅ Aucun doublon détecté.")
else:
    print(f"⚠️ {duplicate_count} doublon(s) détecté(s) — suppression en cours...")
    df = df.drop_duplicates()
    print(f"✅ Dataset après suppression : {df.shape[0]} lignes")

✅ Aucun doublon détecté.


---
## 5. Filtrage des années hors période d'enquête

Les enquêtes couvrent 2016 à 2018 (une année par pays) : toute autre année est une erreur de saisie.

In [9]:
invalid_years = df[~df["year"].isin(SURVEY_YEARS)]

if invalid_years.empty:
    print("✅ Aucune année hors période détectée.")
else:
    print(f"⚠️ {len(invalid_years)} ligne(s) avec une année hors période — suppression en cours...")
    display(invalid_years[["country", "year", "uniqueid"]])
    df = df[df["year"].isin(SURVEY_YEARS)]
    print(f"✅ Dataset après filtrage : {df.shape[0]} lignes")

⚠️ 3 ligne(s) avec une année hors période — suppression en cours...


,country,year,uniqueid
213,Kenya,2029,uniqueid_214
308,Kenya,2056,uniqueid_309
7014,Rwanda,2039,uniqueid_947


✅ Dataset après filtrage : 23521 lignes


---
## 6. Normalisation des noms de colonnes

Standardisation en `snake_case` : suppression des espaces, mise en minuscules.

In [10]:
print("Colonnes avant normalisation :")
print(df.columns.tolist())

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nColonnes après normalisation :")
print(df.columns.tolist())

Colonnes avant normalisation :
['country', 'year', 'uniqueid', 'Has a Bank account', 'Type of Location', 'Cell Phone Access', 'household_size', 'Respondent Age', 'gender_of_respondent', 'The relathip with head', 'marital_status', 'Level of Educuation', 'Type of Job']

Colonnes après normalisation :
['country', 'year', 'uniqueid', 'has_a_bank_account', 'type_of_location', 'cell_phone_access', 'household_size', 'respondent_age', 'gender_of_respondent', 'the_relathip_with_head', 'marital_status', 'level_of_educuation', 'type_of_job']


---
## 7. Cible inconnue et niveau d'éducation sans libellé

- **Cible inconnue.** 36 lignes n'indiquent pas si la personne a un compte (« Unknown » après le traitement des manquants). Elles ne servent ni à entraîner ni à évaluer le modèle : elles sont retirées.
- **Niveau d'éducation « 6 ».** Une modalité sans libellé, absente de la documentation de l'enquête. Rien ne permet de dire à quel niveau elle correspond : elle est traitée comme une valeur inconnue (« Unknown »).

In [11]:
unknown_target = df["has_a_bank_account"] == "Unknown"
df = df[~unknown_target]
print(f"  {unknown_target.sum()} ligne(s) à cible inconnue retirée(s) → {df.shape[0]} lignes")

unlabeled_education = df["level_of_educuation"] == "6"
df.loc[unlabeled_education, "level_of_educuation"] = "Unknown"
print(f"  {unlabeled_education.sum()} ligne(s) au niveau d'éducation « 6 » → 'Unknown'")

  36 ligne(s) à cible inconnue retirée(s) → 23485 lignes


  27 ligne(s) au niveau d'éducation « 6 » → 'Unknown'


---
## 8. Cohérence entre pays

Les quatre enquêtes ne sont pas harmonisées. Un seul défaut est corrigé, parce qu'une source le confirme : les libellés ville / campagne de la Tanzanie. Les autres constats ne modifient pas les données : ils disent ce qui ne se compare pas d'un pays à l'autre.

- **Un pays, une année.** L'effet du pays et celui de l'année ne se séparent pas.
- **Types d'emploi codés différemment.** Ni agriculture ni emploi public en Tanzanie et en Ouganda, « sans revenu » seulement chez eux, pas d'emploi informel en Ouganda. Les types d'emploi ne se comparent qu'à l'intérieur d'un pays.
- **Ville et campagne inversées en Tanzanie** : corrigé plus bas.
- **Taille du foyer en Tanzanie.** Plus de huit foyers sur dix y comptent une ou deux personnes, et tous les foyers de taille 0 sont tanzaniens : la notion de foyer n'est sans doute pas la même.

In [12]:
print("=== Pays × année ===")
display(pd.crosstab(df["country"], df["year"]))

print("=== Types d'emploi présents dans chaque pays ===")
display(pd.crosstab(df["type_of_job"], df["country"]))

=== Pays × année ===


year,2016,2017,2018
country,,,
Kenya,0,0,6066
Rwanda,8734,0,0
Tanzania,0,6584,0
Uganda,0,0,2101


=== Types d'emploi présents dans chaque pays ===


country,Kenya,Rwanda,Tanzania,Uganda
type_of_job,,,,
Dont Know/Refuse to answer,3,56,0,67
Farming and Fishing,1609,3824,0,0
Formally employed Government,260,127,0,0
Formally employed Private,415,161,368,108
Government Dependent,65,91,84,7
Informally employed,1419,2868,1290,0
No Income,0,0,377,247
Other Income,154,323,34,568
Remittance Dependent,1187,660,674,0


In [13]:
SECONDARY_OR_MORE = ["Secondary education", "Tertiary education", "Vocational/Specialised training"]


def location_profile(data: pd.DataFrame) -> pd.DataFrame:
    """Part des urbains, puis compte, téléphone et éducation selon la localisation, par pays."""
    known = data[data["has_a_bank_account"].isin(["Yes", "No"]) & data["type_of_location"].isin(["Urban", "Rural"])].assign(
        account=lambda d: d["has_a_bank_account"].eq("Yes"),
        phone=lambda d: d["cell_phone_access"].eq("Yes"),
        secondary_or_more=lambda d: d["level_of_educuation"].isin(SECONDARY_OR_MORE),
    )
    print("=== Part des urbains par pays ===")
    display(known.groupby("country")["type_of_location"].apply(lambda s: s.eq("Urban").mean()).round(3))
    profile = known.groupby(["country", "type_of_location"])[["account", "phone", "secondary_or_more"]].mean()
    print("=== Compte, téléphone et éducation selon la localisation, par pays ===")
    display(profile.round(3))
    return profile


profile_before = location_profile(df)

=== Part des urbains par pays ===


country
Kenya       0.416
Rwanda      0.143
Tanzania    0.727
Uganda      0.282
Name: type_of_location, dtype: float64

=== Compte, téléphone et éducation selon la localisation, par pays ===


account  phone  secondary_or_more
country  type_of_location                                   
Kenya    Rural               0.173  0.732              0.312
         Urban               0.359  0.868              0.560
Rwanda   Rural               0.083  0.812              0.158
         Urban               0.308  0.939              0.458
Tanzania Rural               0.207  0.794              0.359
         Urban               0.048  0.524              0.127
Uganda   Rural               0.042  0.641              0.216
         Urban               0.197  0.840              0.538

### Ville et campagne : libellés inversés en Tanzanie

Dans le fichier, 72,7 % des répondants tanzaniens sont « urbains », contre 14 à 42 % dans les autres pays. Et ces « urbains » ont moins souvent un compte, un téléphone et au moins le secondaire que les « ruraux » : l'inverse des trois autres pays.

Le rapport officiel de l'enquête tranche : *FinScope Tanzania 2017*, p. 14, « Chart 1: Urban vs Rural Split — 34% Urban, 66% Rural » ([rapport FSDT](https://www.fsdt.or.tz/wp-content/uploads/2017/09/FinScope-Tanzania-2017-Insights-that-Drive-Innovation.pdf)). Le même rapport note que les adultes des zones rurales sont plus souvent exclus des services financiers (p. 57).

Les libellés « Urban » et « Rural » de la Tanzanie sont donc échangés. Les valeurs « Unknown » ne bougent pas.

In [14]:
LOCATION_INVERTED_COUNTRIES = ["Tanzania"]
LOCATION_SWAP = {"Urban": "Rural", "Rural": "Urban"}

rows = df["country"].isin(LOCATION_INVERTED_COUNTRIES) & df["type_of_location"].isin(LOCATION_SWAP)
df.loc[rows, "type_of_location"] = df.loc[rows, "type_of_location"].replace(LOCATION_SWAP)
print(f"  {rows.sum()} lignes de {', '.join(LOCATION_INVERTED_COUNTRIES)} : 'Urban' et 'Rural' échangés ('Unknown' inchangé)")

profile_after = location_profile(df)

# Après correction, dans chaque pays, les urbains ont plus souvent un compte, un téléphone et au moins le secondaire.
gaps = profile_after.xs("Urban", level="type_of_location") - profile_after.xs("Rural", level="type_of_location")
assert (gaps > 0).all().all(), gaps
print("\n✅ Dans les quatre pays, les urbains ont plus souvent un compte, un téléphone et au moins le secondaire")

  6577 lignes de Tanzania : 'Urban' et 'Rural' échangés ('Unknown' inchangé)
=== Part des urbains par pays ===


country
Kenya       0.416
Rwanda      0.143
Tanzania    0.273
Uganda      0.282
Name: type_of_location, dtype: float64

=== Compte, téléphone et éducation selon la localisation, par pays ===


account  phone  secondary_or_more
country  type_of_location                                   
Kenya    Rural               0.173  0.732              0.312
         Urban               0.359  0.868              0.560
Rwanda   Rural               0.083  0.812              0.158
         Urban               0.308  0.939              0.458
Tanzania Rural               0.048  0.524              0.127
         Urban               0.207  0.794              0.359
Uganda   Rural               0.042  0.641              0.216
         Urban               0.197  0.840              0.538


✅ Dans les quatre pays, les urbains ont plus souvent un compte, un téléphone et au moins le secondaire


In [15]:
print("=== Taille du foyer : part des foyers de 0 à 2 personnes, par pays ===")
display(df.groupby("country")["household_size"].apply(lambda s: s.le(2).mean()).round(3))

print("=== Foyers de taille 0 ===")
display(df.loc[df["household_size"] == 0, "country"].value_counts())

=== Taille du foyer : part des foyers de 0 à 2 personnes, par pays ===


country
Kenya       0.306
Rwanda      0.157
Tanzania    0.828
Uganda      0.203
Name: household_size, dtype: float64

=== Foyers de taille 0 ===


country
Tanzania    36
Name: count, dtype: int64

---
## 9. État final et contrôles qualité

Les corrections de ce notebook sont transformées en contrôles automatiques (`src/quality_checks.py`), relancés ici avant l'export et avant chaque entraînement du modèle (`python -m src.model`). Un défaut qui revient arrête l'export au lieu de passer.

In [16]:
print(f"✅ Dataset nettoyé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.info()
df.head()

✅ Dataset nettoyé : 23485 lignes × 13 colonnes
<class 'pandas.core.frame.DataFrame'>
Index: 23485 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country                 23485 non-null  object 
 1   year                    23485 non-null  int64  
 2   uniqueid                23485 non-null  object 
 3   has_a_bank_account      23485 non-null  object 
 4   type_of_location        23485 non-null  object 
 5   cell_phone_access       23485 non-null  object 
 6   household_size          23485 non-null  float64
 7   respondent_age          23485 non-null  float64
 8   gender_of_respondent    23485 non-null  object 
 9   the_relathip_with_head  23485 non-null  object 
 10  marital_status          23485 non-null  object 
 11  level_of_educuation     23485 non-null  object 
 12  type_of_job             23485 non-null  object 
dtypes: float64(2), int64(1), object(10)
memory usage:

,country,year,uniqueid,has_a_bank_account,type_of_location,cell_phone_access,household_size,respondent_age,gender_of_respondent,the_relathip_with_head,marital_status,level_of_educuation,type_of_job
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3.0,24.0,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5.0,70.0,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5.0,26.0,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5.0,34.0,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8.0,26.0,Male,Child,Single/Never Married,Primary education,Informally employed


In [17]:
import sys

sys.path.insert(0, "..")
from src.quality_checks import assert_clean_data, check_clean_data

for name, issues in check_clean_data(df).items():
    print(f"{'✅' if not issues else '❌'} {name}" + (f" : {issues}" if issues else ""))
assert_clean_data(df)

✅ Aucune valeur manquante
✅ Années d'enquête de 2016 à 2018
✅ Une seule année d'enquête par pays
✅ Aucun pays inconnu
✅ Cible connue (Yes ou No)
✅ Aucun doublon
✅ Niveaux d'éducation nommés
✅ Urbains plus souvent titulaires d'un compte


---
## 10. Export du dataset nettoyé

In [18]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Dataset exporté : {OUTPUT_PATH}")

✅ Dataset exporté : ../data/financial_inclusion_clean.csv
